Universidad del Valle de Guatemala  
Departamento de Ciencias de la Computación  
CC3084 - Data Science - sección 40

Integrantes:  
- Cristian Túnchez (231359)
- Dulce Ambrosio (231143)
- Daniel Chet (231177)
- Javier Linares (231135)

# 04. Afinado de un transformer

**MAP - Charting Student Math Misunderstandings**

En este Notebook, el propósito es contrastar la representación densa y contextual de un transformer contra las representaciones de frecuencia del cuaderno anterior. Nos interesa especialmente comprobar si la dificultad reportada por los organizadores de la competencia, que atribuyen el fracaso de los modelos preentrenados a la complejidad del contenido matemático, se reproduce aquí.

## 0. Arranque

Este cuaderno funciona igual en **Google Colab** y en una instalación local.

En Colab la celda siguiente clona el repositorio, instala las dependencias y deja el proyecto listo. La guía de configuración en Colab está en el [README](../README.md) principal.

En local la celda no hace nada salvo ubicar la raíz del proyecto.

In [2]:
import os
import subprocess
import sys
from pathlib import Path

EN_COLAB = "google.colab" in sys.modules
REPOSITORIO = "https://github.com/Tunchxz/math-misconception-map.git"
CARPETA = "math-misconception-map"

if EN_COLAB:
    if not Path(CARPETA).exists():
        subprocess.run(["git", "clone", "--depth", "1", "-q", REPOSITORIO, CARPETA], check=True)
    os.chdir(f"/content/{CARPETA}")
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True
    )
    try:
        from google.colab import userdata

        os.environ["KAGGLE_ACCESS_TOKEN"] = userdata.get("KAGGLE_ACCESS_TOKEN")
        print("Token de Kaggle cargado desde los secretos de Colab.")
    except Exception:
        print("Registre el secreto KAGGLE_ACCESS_TOKEN para poder descargar los datos.")

RAIZ = Path.cwd() if (Path.cwd() / "src").exists() else Path.cwd().parent
if str(RAIZ) not in sys.path:
    sys.path.insert(0, str(RAIZ))

print("Entorno:", "Google Colab" if EN_COLAB else "local")
print("Raíz del proyecto:", RAIZ)

Token de Kaggle cargado desde los secretos de Colab.
Entorno: Google Colab
Raíz del proyecto: /content/math-misconception-map


## 1. Preparación del entorno

La primera celda detecta si el cuaderno corre en Colab y, en ese caso, instala las dependencias
y descarga los datos. En local no hace nada, de modo que el cuaderno funciona en ambos sitios.

In [3]:
# El afinado requiere bibliotecas que no forman parte de requirements.txt,
# porque solo se usan en este cuaderno.
import subprocess
import sys

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q",
     "transformers>=4.40,<5", "datasets>=2.19", "accelerate>=0.30"],
    check=True,
)

import numpy as np
import pandas as pd
import torch

print("PyTorch:", torch.__version__)
print("GPU disponible:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Dispositivo:", torch.cuda.get_device_name(0))
    print("Memoria:", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), "GB")
else:
    print("ATENCIÓN: sin GPU el afinado tardará horas.")
    print("En Colab: Entorno de ejecución > Cambiar tipo de entorno de ejecución > T4 GPU")

PyTorch: 2.11.0+cu130
GPU disponible: True
Dispositivo: Tesla T4
Memoria: 15.6 GB


## 2. Obtención de los datos

En Colab se descarga el conjunto limpio desde el repositorio, o bien se sube manualmente el
archivo `train_limpio.parquet` que produjo el cuaderno 01. En local se lee de `data/processed`.

In [4]:
from src.data.make_dataset import asegurar_datos

train = asegurar_datos()
print("Observaciones:", f"{len(train):,}")
train.head(3)

Conjunto limpio encontrado en /content/math-misconception-map/data/processed/train_limpio.parquet
Observaciones: 36,696


,row_id,QuestionId,QuestionText,MC_Answer,MC_Answer_limpia,StudentExplanation,explicacion_limpia,explicacion_min,Category,respuesta_correcta,...,n_caracteres,n_palabras,long_media_palabra,n_digitos,tiene_digitos,tiene_fraccion,tiene_simbolo_mat,tiene_interrogacion,tiene_porque,respuesta_correcta_pregunta
0,0,31772,What fraction of the shape is not shaded? Give...,\( \frac{1}{3} \),1/3,0ne third is equal to tree nineth,0ne third is equal to tree nineth,0ne third is equal to tree nineth,True_Correct,True,...,33,7,4.71,1,True,False,False,False,False,\( \frac{1}{3} \)
1,1,31772,What fraction of the shape is not shaded? Give...,\( \frac{1}{3} \),1/3,1 / 3 because 6 over 9 is 2 thirds and 1 third...,1 / 3 because 6 over 9 is 2 thirds and 1 third...,1 / 3 because 6 over 9 is 2 thirds and 1 third...,True_Correct,True,...,61,16,3.81,6,True,True,False,False,True,\( \frac{1}{3} \)
2,2,31772,What fraction of the shape is not shaded? Give...,\( \frac{1}{3} \),1/3,"1 3rd is half of 3 6th, so it is simplee to un...","1 3rd is half of 3 6th, so it is simplee to un...","1 3rd is half of 3 6th, so it is simplee to un...",True_Neither,True,...,55,13,4.23,4,True,False,False,False,False,\( \frac{1}{3} \)


In [5]:
SEMILLA = 42
SIN_MISCONCEPTION = "NA"
ALIAS = {"Wrong_Fraction": "Wrong_fraction"}

train["misconception_unificada"] = train["misconception"].map(lambda v: ALIAS.get(v, v))
train["objetivo_unificado"] = train["Category"] + ":" + train["misconception_unificada"]

np.random.seed(SEMILLA)
torch.manual_seed(SEMILLA)

print("Clases de Category:", train["Category"].nunique())
print("Errores conceptuales:", train["misconception_unificada"].nunique() - 1)

Clases de Category: 6
Errores conceptuales: 34


## 3. Las dos formas de entrada

Se comparan dos maneras de presentar el texto al modelo.

- **Solo la explicación.** El modelo ve únicamente lo que escribió el estudiante.
- **Explicación con contexto.** Se antepone la respuesta seleccionada, separada por el token
  especial del modelo.

La comparación importa porque el análisis exploratorio mostró que el contexto de la pregunta
está casi perfectamente asociado con la etiqueta. Dar demasiado contexto puede inducir al modelo
a memorizar en lugar de razonar sobre el texto.

In [6]:
def construir_entradas(df, con_contexto, separador):
    '''Devuelve la lista de textos de entrada del modelo.'''
    explicacion = df["explicacion_limpia"].astype(str)
    if not con_contexto:
        return explicacion.tolist()
    return (df["MC_Answer_limpia"].astype(str) + f" {separador} " + explicacion).tolist()


print("Sin contexto:", construir_entradas(train.head(2), False, "[SEP]"))
print("Con contexto:", construir_entradas(train.head(2), True, "[SEP]"))

Sin contexto: ['0ne third is equal to tree nineth', '1 / 3 because 6 over 9 is 2 thirds and 1 third is not shaded.']
Con contexto: ['1/3 [SEP] 0ne third is equal to tree nineth', '1/3 [SEP] 1 / 3 because 6 over 9 is 2 thirds and 1 third is not shaded.']


## 4. Partición por pregunta

Se usa el esquema agrupado, que deja preguntas completas fuera del entrenamiento. Es el más
exigente y el único que refleja lo que la competencia evalúa, de modo que el resultado es
comparable con el del cuaderno anterior.

In [7]:
from sklearn.model_selection import GroupKFold

gkf = GroupKFold(n_splits=5)
idx_tr, idx_te = next(iter(gkf.split(train, train["Category"], train["QuestionId"])))

entrenamiento = train.iloc[idx_tr].reset_index(drop=True)
prueba = train.iloc[idx_te].reset_index(drop=True)

print(f"Entrenamiento: {len(entrenamiento):,} filas, "
      f"{entrenamiento['QuestionId'].nunique()} preguntas")
print(f"Prueba:        {len(prueba):,} filas, {prueba['QuestionId'].nunique()} preguntas")
print("Preguntas compartidas:",
      len(set(entrenamiento['QuestionId']) & set(prueba['QuestionId'])))

Entrenamiento: 29,010 filas, 12 preguntas
Prueba:        7,686 filas, 3 preguntas
Preguntas compartidas: 0


## 5. Afinado del modelo

Se emplea **MiniLM-L12**, una versión destilada de BERT con unos 33 millones de parámetros.
Wang et al. (2020) la propusieron comprimiendo las relaciones de autoatención del modelo
original. Su tamaño reducido permite entrenarla en los recursos gratuitos de Colab y, sobre
todo, cargarla dentro de la aplicación interactiva del proyecto.

In [8]:
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    DataCollatorWithPadding,
    Trainer,
    TrainingArguments,
)
from datasets import Dataset
from sklearn.metrics import f1_score
from sklearn.preprocessing import LabelEncoder

MODELO_BASE = "microsoft/MiniLM-L12-H384-uncased"
LONGITUD_MAX = 128
EPOCAS = 3
LOTE = 32

tokenizador = AutoTokenizer.from_pretrained(MODELO_BASE)
print("Token separador:", tokenizador.sep_token)
print("Vocabulario:", tokenizador.vocab_size)

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/2.00 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/385 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

Token separador: [SEP]
Vocabulario: 30522


In [9]:
def preparar_dataset(df, codificador, con_contexto):
    '''Tokeniza los textos y adjunta las etiquetas codificadas.'''
    textos = construir_entradas(df, con_contexto, tokenizador.sep_token)
    ds = Dataset.from_dict(
        {"text": textos, "labels": codificador.transform(df["Category"])}
    )
    return ds.map(
        lambda lote: tokenizador(lote["text"], truncation=True, max_length=LONGITUD_MAX),
        batched=True,
        remove_columns=["text"],
    )


def metricas(pred):
    '''Calcula F1 macro y ponderado a partir de la salida del Trainer.'''
    etiquetas = pred.label_ids
    predichas = pred.predictions.argmax(-1)
    return {
        "f1_macro": f1_score(etiquetas, predichas, average="macro", zero_division=0),
        "f1_ponderado": f1_score(etiquetas, predichas, average="weighted", zero_division=0),
    }

In [10]:
def afinar(con_contexto, etiqueta_experimento):
    '''Afina el modelo con la forma de entrada indicada y devuelve sus métricas.'''
    codificador = LabelEncoder().fit(train["Category"])
    ds_tr = preparar_dataset(entrenamiento, codificador, con_contexto)
    ds_te = preparar_dataset(prueba, codificador, con_contexto)

    modelo = AutoModelForSequenceClassification.from_pretrained(
        MODELO_BASE, num_labels=len(codificador.classes_)
    )

    args = TrainingArguments(
        output_dir=f"salida_{etiqueta_experimento}",
        num_train_epochs=EPOCAS,
        per_device_train_batch_size=LOTE,
        per_device_eval_batch_size=LOTE * 2,
        learning_rate=3e-5,
        warmup_ratio=0.1,
        weight_decay=0.01,
        eval_strategy="epoch",
        save_strategy="no",
        logging_steps=100,
        seed=SEMILLA,
        fp16=torch.cuda.is_available(),
        report_to=[],
    )

    entrenador = Trainer(
        model=modelo,
        args=args,
        train_dataset=ds_tr,
        eval_dataset=ds_te,
        data_collator=DataCollatorWithPadding(tokenizador),
        compute_metrics=metricas,
    )

    import time
    t0 = time.perf_counter()
    entrenador.train()
    segundos = time.perf_counter() - t0

    evaluacion = entrenador.evaluate()
    return {
        "experimento": etiqueta_experimento,
        "f1_macro": evaluacion["eval_f1_macro"],
        "f1_ponderado": evaluacion["eval_f1_ponderado"],
        "segundos": segundos,
    }, entrenador, codificador

In [11]:
resultado_sin, entrenador_sin, codificador = afinar(False, "solo_explicacion")
print(resultado_sin)

Map:   0%|          | 0/29010 [00:00<?, ? examples/s]

Map:   0%|          | 0/7686 [00:00<?, ? examples/s]

pytorch_model.bin:   0%|          | 0.00/133M [00:00<?, ?B/s]

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at microsoft/MiniLM-L12-H384-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


model.safetensors:   0%|          | 0.00/133M [00:00<?, ?B/s]

Epoch,Training Loss,Validation Loss,F1 Macro,F1 Ponderado
1,1.003200,1.013989,0.287335,0.586630
2,0.816200,1.303314,0.304353,0.550664
3,0.754100,1.262351,0.304967,0.558117


{'experimento': 'solo_explicacion', 'f1_macro': 0.3049667665842366, 'f1_ponderado': 0.5581167852307982, 'segundos': 193.76473597999995}


In [12]:
resultado_con, entrenador_con, _ = afinar(True, "con_contexto")
print(resultado_con)

Map:   0%|          | 0/29010 [00:00<?, ? examples/s]

Map:   0%|          | 0/7686 [00:00<?, ? examples/s]

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at microsoft/MiniLM-L12-H384-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Epoch,Training Loss,Validation Loss,F1 Macro,F1 Ponderado
1,0.631700,1.594563,0.296590,0.549526
2,0.483100,2.849158,0.225000,0.378421
3,0.433500,2.792719,0.269074,0.479598


{'experimento': 'con_contexto', 'f1_macro': 0.2690744995539322, 'f1_ponderado': 0.47959798979895657, 'segundos': 144.05503107999994}


## 6. Comparación de las dos formas de entrada

In [13]:
comparacion = pd.DataFrame([resultado_sin, resultado_con])
comparacion["f1_macro"] = comparacion["f1_macro"].round(4)
comparacion["f1_ponderado"] = comparacion["f1_ponderado"].round(4)
comparacion["minutos"] = (comparacion["segundos"] / 60).round(1)
comparacion[["experimento", "f1_macro", "f1_ponderado", "minutos"]]

,experimento,f1_macro,f1_ponderado,minutos
0,solo_explicacion,0.3050,0.5581,3.2
1,con_contexto,0.2691,0.4796,2.4


## 7. Guardado del modelo

Se guarda la mejor de las dos variantes, en el formato que la aplicación puede cargar.

In [15]:
mejor = entrenador_con if resultado_con["f1_macro"] >= resultado_sin["f1_macro"] else entrenador_sin
etiqueta = (
    "con_contexto" if resultado_con["f1_macro"] >= resultado_sin["f1_macro"]
    else "solo_explicacion"
)

destino = Path("modelo_minilm")
mejor.save_model(destino)
tokenizador.save_pretrained(destino)
pd.Series(codificador.classes_).to_csv(destino / "clases.csv", index=False, header=False)
comparacion.to_csv(destino / "metricas.csv", index=False)

print("Variante guardada:", etiqueta)
print("Archivos:", sorted(p.name for p in destino.iterdir()))
tamano = sum(p.stat().st_size for p in destino.rglob("*")) / 1e6
print(f"Tamaño total: {tamano:.1f} MB")

if "google.colab" in sys.modules:
    import shutil

    shutil.make_archive("modelo_minilm", "zip", destino)
    from google.colab import files

    files.download("modelo_minilm.zip")
    print("Descomprima el archivo dentro de models/ del repositorio local.")

Variante guardada: solo_explicacion
Archivos: ['clases.csv', 'config.json', 'metricas.csv', 'model.safetensors', 'special_tokens_map.json', 'tokenizer.json', 'tokenizer_config.json', 'training_args.bin', 'vocab.txt']
Tamaño total: 134.4 MB


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Descomprima el archivo dentro de models/ del repositorio local.
